In [37]:
!pip install Groq
!pip install python-dotenv
!pip install trafilatura

DEPRECATION: Loading egg at c:\users\jyjj0\appdata\local\programs\python\python311\lib\site-packages\py_hanspell-1.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


DEPRECATION: Loading egg at c:\users\jyjj0\appdata\local\programs\python\python311\lib\site-packages\py_hanspell-1.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


DEPRECATION: Loading egg at c:\users\jyjj0\appdata\local\programs\python\python311\lib\site-packages\py_hanspell-1.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import os
import json
import requests
from typing import List, Dict
from groq import Groq
from datetime import datetime
import time
from dotenv import load_dotenv
import trafilatura

In [39]:
load_dotenv(dotenv_path=".env")  # .env 파일 로드

SERP_API_KEY = os.getenv("SERP_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

serp_api_key = os.getenv("SERP_API_KEY")

print("API Key:", serp_api_key)

API Key: e3e4c249d0ebb39053746c12d0238e287ccc82c663e32ecc7b171a57182bd907


In [40]:
# (1) 쿼리 생성 함수
def get_queries(company_name: str, industry: str):
    base_queries = {
        "산업 전망": f"{company_name} {industry} 산업 전망",
        "규제": f"{company_name} {industry} 규제",
        "연구개발": f"{company_name} {industry} 연구개발",
        "시장 동향": f"{company_name} {industry} 시장 동향"
    }
    return base_queries

In [41]:
# (2) 뉴스 검색
def search_news(query: str, serp_api_key: str, num_results: int = 5):
    url = "https://serpapi.com/search"
    params = {
        "engine": "google_news",
        "q": query,
        "api_key": serp_api_key,
        "num": num_results,
        "hl": "ko",  
        "gl": "kr",  
        "sort": "date"  
    }
    response = requests.get(url, params=params)
    results = response.json()

    articles = []
    for item in results.get("news_results", []):
        articles.append({
            "title": item.get("title"),
            "link": item.get("link")
        })
    return articles

    try:
        print(f"🔍 검색 중: {query}")
        response = requests.get(url, params=params)
        response.raise_for_status()  # HTTP 에러 체크
        results = response.json()
        
        # API 에러 체크
        if "error" in results:
            print(f"❌ SerpAPI 에러: {results['error']}")
            return []
        
        articles = []
        news_results = results.get("news_results", [])
        
        if not news_results:
            print(f"⚠️  '{query}' 검색 결과 없음")
            return []
        
        for item in news_results[:num_results]:
            articles.append({
                "title": item.get("title", "제목 없음"),
                "link": item.get("link", ""),
                "source": item.get("source", "출처 없음"),
                "date": item.get("date", "날짜 없음")
            })
        
        print(f"✅ {len(articles)}개 기사 발견")
        return articles
        
    except requests.RequestException as e:
        print(f"❌ 네트워크 에러: {e}")
        return []
    except Exception as e:
        print(f"❌ 예상치 못한 에러: {e}")
        return []

In [42]:
# (3) 본문 크롤링
def crawl_articles(articles: List[Dict]) -> List[Dict]:
    crawled = []
    
    for i, article in enumerate(articles, 1):
        print(f"📰 크롤링 중 ({i}/{len(articles)}): {article['title'][:50]}...")
        
        try:
            # 요청 간격 조절 (서버 부하 방지)
            time.sleep(1)
            
            downloaded = trafilatura.fetch_url(article["link"])
            if downloaded:
                text = trafilatura.extract(downloaded)
                if text and len(text.strip()) > 100:  # 최소 길이 체크
                    crawled.append({
                        "title": article["title"],
                        "link": article["link"],
                        "source": article["source"],
                        "date": article["date"],
                        "content": text
                    })
                    print("✅ 크롤링 성공")
                else:
                    print("⚠️  본문 추출 실패 (내용 부족)")
                    print(f"링크: {article['link']}")
            else:
                print("⚠️  페이지 다운로드 실패")
                
        except Exception as e:
            print(f"❌ 크롤링 에러: {e}")
            continue
    
    print(f"\n📊 총 {len(crawled)}개 기사 크롤링 완료")
    return crawled


In [43]:
# (4) 사업위험 작성 지침 (고정값)
RISK_GUIDELINES = """
증권신고서 사업위험 작성 지침:

5-1 (시장환경 변화): 사업의 시장환경이 급속히 변하는 경우 향후 영업실적 및 재무구조에 미칠 영향
- 5-1-1: 중요한 법적 규제 신설이나 정부 정책 시행의 부정적 영향
- 5-1-3: 산업 기술의 급변으로 인한 영업 영향 (제품수명주기, 연구개발비, 기술변화 적응 실패 위험)

5-2 (매출․매입 편중): 특정 제품․거래처 등에 편중으로 인한 위험
- 5-2-1: 소수 고객 의존으로 인한 매출액/영업이익 급변 위험
- 5-2-3: 소수 공급자 의존으로 인한 매입․생산 차질 위험

5-4 (연구개발): 연구개발이 중요한 사업의 경우
- 5-4-1: 개발 완료 기술의 상품화 가능성이 낮을 경우
- 5-4-4: 소수 핵심인력 의존도가 높을 경우

5-7 (신규 사업): 새로 진출하는 사업이 영업성과에 중대한 영향을 미치는 경우
- 5-7-1: 신규 사업 경쟁력 확보를 위한 기술력, 자본력, 인력 부족
- 5-7-2: 신규 사업 대규모 자금 소요 시 자금조달 위험

각 위험요소는 구체적 상황, 부정적 영향, 대응방안을 포함하여 줄글로 작성해야 함.
"""

In [44]:
# (5) LLM 분석 (Groq API 사용)
def analyze_risks_with_groq(crawled_articles: List[Dict], company_name: str, industry: str, groq_api_key: str) -> str:
    try:
        from groq import Groq
        
        client = Groq(api_key=groq_api_key)
        
        # 프롬프트 구성
        articles_text = ""
        for article in crawled_articles:
            articles_text += f"\n📰 {article['title']}\n출처: {article['source']}\n날짜: {article['date']}\n내용: {article['content'][:1000]}...\n"
        
        prompt = f"""
{RISK_GUIDELINES}

위 지침에 따라 '{company_name}' ({industry} 산업)의 사업위험을 분석하세요.

다음 뉴스 기사들을 참고하여 각 위험요소별로 구체적인 사업위험을 작성해주세요:
{articles_text}

요구사항:
1. 5-1, 5-2, 5-4, 5-7 항목 중 해당되는 것들을 선별
2. 각 항목마다 현재 상황, 위험 내용, 예상 영향, 대응방안을 포함한 문단 형태로 작성
3. 뉴스 내용을 구체적으로 인용하여 현실적인 위험 분석 제공
4. 증권신고서 형식에 맞는 전문적 문체 사용
5. 한글로 작성해줘
"""

        print("🤖 AI 분석 중...")
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama3-8b-8192",
            temperature=0.3,
            max_tokens=2000
        )
        
        return chat_completion.choices[0].message.content
        
    except ImportError:
        print("❌ groq 라이브러리가 설치되지 않았습니다. 'pip install groq' 실행하세요.")
        return "Groq 라이브러리 설치 필요"
    except Exception as e:
        print(f"❌ LLM 분석 에러: {e}")
        return f"분석 실패: {e}"

In [ ]:
# (A) 기사 요약 함수
def summarize_article(article_content: str, groq_api_key: str) -> str:
    """기사 본문을 3줄 요약"""
    client = Groq(api_key=groq_api_key)
    prompt = f"""
다음 기사를 3줄로 요약하세요. 핵심 내용만 남기고, 불필요한 수식어나 배경설명은 빼세요.

기사:
{article_content[:1500]}  # 기사 본문이 너무 길면 일부만
"""
    try:
        completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="gemma2-9b-it",
            temperature=0.3,
            #max_tokens=
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ 요약 실패: {e}")
        return "요약 실패"


# (B) 주제별 분석 함수
def analyze_topic(summaries: list, topic: str, company_name: str, industry: str, groq_api_key: str) -> str:
    """요약 기사 묶음을 기반으로 주제별 사업위험 요소 분석"""
    client = Groq(api_key=groq_api_key)
    summaries_text = "\n".join([f"- {s}" for s in summaries])

    prompt = f"""
증권신고서 사업위험 작성 지침({RISK_GUIDELINES})에 따라 
'{company_name}' ({industry}) 산업의 '{topic}' 관련 사업위험을 분석하세요.

참고 기사 요약:
{summaries_text}

요구사항:
1. 관련된 위험요소(5-1, 5-2, 5-4, 5-7 중 해당되는 것) 선택
2. 현재 상황, 위험 내용, 예상 영향, 대응방안을 포함
3. 증권신고서 형식에 맞는 전문적 문체 사용
4. 한글로 작성해주세요
"""

    try:
        completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama3-8b-8192",
            temperature=0.3,
            max_tokens=800
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ 분석 실패: {e}")
        return "분석 실패"

# 최졷 통합 보고서(결과)
def summarize_final_risks(topic_results: dict, company_name: str, industry: str, groq_api_key: str) -> str:
    """주제별 분석 결과를 합쳐 최종 사업위험 보고서 작성"""
    client = Groq(api_key=groq_api_key)
    topic_text = "\n\n".join([f"[{topic}]\n{result}" for topic, result in topic_results.items()])

    prompt = f"""
다음은 '{company_name}' ({industry}) 산업 관련 사업위험 분석 결과입니다. 
각 주제별 분석을 종합해 증권신고서 사업위험(제5항) 섹션 초안을 작성하세요.

주제별 분석:
{topic_text}

작성 지침:
1. 사업위험은 5-1, 5-2, 5-4, 5-7 항목 중 해당되는 것만 포함
2. 각 항목은 다음과 같은 제목 형식을 사용:
   - 5-1 → 가. 시장환경 변화
   - 5-2 → 나. 매출·매입 편중
   - 5-4 → 다. 연구개발
   - 5-7 → 라. 신규 사업
3. 각 항목은 (1)(2)(3)… 세부위험요소를 넣고, 그 아래 문단으로 구체적으로 서술
4. 각 문단에는 반드시 현재 상황, 위험 내용, 예상 영향, 대응방안을 포함
5. 불필요한 수식어 없이, 전문적이고 간결한 증권신고서 문체 사용
6. 주제별 분석 결과에 중복된 위험요소가 있으면 하나로 통합하여 정리
7. 한글로 작성
"""

    completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama3-8b-8192",
        temperature=0.3,
        max_tokens=2000
    )
    return completion.choices[0].message.content.strip()


In [46]:
# (6) 결과 저장
def save_results(company_name: str, industry: str, crawled_articles: List[Dict], risk_analysis: str):
    results = {
        "company_info": {
            "name": company_name,
            "industry": industry,
            "analysis_date": time.strftime("%Y-%m-%d %H:%M:%S")
        },
        "articles": crawled_articles,
        "risk_analysis": risk_analysis
    }
    
    filename = f"{company_name}_사업위험분석_{time.strftime('%Y%m%d')}.json"
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"💾 결과 저장: {filename}")
    
    # 위험분석만 별도 텍스트 파일로 저장
    risk_filename = f"{company_name}_사업위험_{time.strftime('%Y%m%d')}.txt"
    with open(risk_filename, 'w', encoding='utf-8') as f:
        f.write(f"[{company_name} 사업위험 분석]\n")
        f.write(f"산업: {industry}\n")
        f.write(f"분석일시: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(risk_analysis)
    
    print(f"📄 사업위험 분석서 저장: {risk_filename}")


In [47]:
def main():
    print("🚀 뉴스 기반 사업위험 분석 시스템 시작\n")

    # (0) API 키 체크
    if not SERP_API_KEY:
        print("❌ SerpAPI 키를 설정해주세요!")
        return
    if not GROQ_API_KEY:
        print("❌ Groq API 키를 설정해주세요!")
        return

    # (1) 회사 정보 입력
    company_name = "티앤알바이오팹"
    industry = input("해당 회사의 산업을 입력하세요 (예: 반도체, 자동차, 금융): ")

    # (2) 주제별 기사 수집
    queries = get_queries(company_name, industry)
    topic_crawled = {}
    for query in queries:
        articles = search_news(query, SERP_API_KEY, num_results=5)
        crawled = crawl_articles(articles) if articles else []
        topic_crawled[query] = crawled
        time.sleep(2)

    # (3) 전체 기사 중복 제거
    all_crawled = [a for crawled in topic_crawled.values() for a in crawled]
    unique_crawled, seen = [], set()
    for article in all_crawled:
        if article['link'] not in seen:
            unique_crawled.append(article)
            seen.add(article['link'])

    if not unique_crawled:
        print("❌ 크롤링된 기사가 없어 분석을 진행할 수 없습니다.")
        return

    # (4) 주제별 요약 & 분석
    topic_results = {}
    for topic, crawled in topic_crawled.items():
        summaries = []
        for article in crawled:
            summary = summarize_article(article['content'], GROQ_API_KEY)
            summaries.append(f"{article['title']} ({article['date']}, {article['source']}): {summary}")
        analysis = analyze_topic(summaries, topic, company_name, industry, GROQ_API_KEY)
        topic_results[topic] = analysis

    # (5) 최종 통합 보고서
    final_risk_report = summarize_final_risks(topic_results, company_name, industry, GROQ_API_KEY)

    print("\n📋 최종 사업위험 보고서")
    print("="*60)
    print(final_risk_report)

    # (6) 저장
    save_results(company_name, industry, unique_crawled, final_risk_report)
    print("\n✅ 분석 완료!")


if __name__ == "__main__":
    main()

🚀 뉴스 기반 사업위험 분석 시스템 시작

📰 크롤링 중 (1/100): 미국 가는 산업장관…조선·원자력 협력 방안 등 논의 전망...
❌ 크롤링 에러: 'source'
📰 크롤링 중 (2/100): 정원워케이션×잔망루피 콜라보 대성공…순천시 캐릭터 산업 전망 밝혀...
❌ 크롤링 에러: 'source'
📰 크롤링 중 (3/100): 2025년 하반기 13대 주력산업 전망...
⚠️  페이지 다운로드 실패
📰 크롤링 중 (4/100): 2040년 국내 건설 수주 300조 돌파 전망… 건산연 “대형 국책사업 본격화” - 조선비...
⚠️  본문 추출 실패 (내용 부족)
링크: https://biz.chosun.com/real_estate/real_estate_general/2025/08/18/CSB3T27VSBB5RAGSUZMS4X4L4U/
📰 크롤링 중 (5/100): 6·27 규제 '직격탄'…8월 서울 주택사업 전망, 전월比 '반토막'...
❌ 크롤링 에러: 'source'
📰 크롤링 중 (6/100): [시장 전망] 석유화학, 2032년 9562억 달러 규모로 성장...
❌ 크롤링 에러: 'source'
📰 크롤링 중 (7/100): 6·27 대출 규제에 주택사업 전망지수 급락…비관적 전망 확대...
❌ 크롤링 에러: 'source'
📰 크롤링 중 (8/100): 6·27 대출 규제에…8월 수도권 주택사업 전망 ‘먹구름’...


KeyboardInterrupt: 

In [ ]:
topic_results = {}

for topic, crawled in topic_results.items():
    print(f"\n📰 {topic} 기사 요약 중...")

    # 1. 기사 요약
    summaries = []
    for article in crawled:
        summary = summarize_article(article['content'], GROQ_API_KEY)
        summaries.append(f"{article['title']} ({article['date']}, {article['source']}): {summary}")

    # 2. 주제별 분석
    analysis = analyze_topic(summaries, topic, company_name, industry, GROQ_API_KEY)
    topic_results[topic] = analysis

# 3. 최종 통합 보고서
final_risk_report = summarize_final_risks(topic_results, company_name, industry, GROQ_API_KEY)

print("\n📋 최종 사업위험 보고서")
print("="*60)
print(final_risk_report)


NameError: name 'topic_crawled' is not defined